# Lesson 04 - टूल उपयोग डिज़ाइन पैटर्न

इस पाठ में, आप Microsoft Agent Framework (Python) का उपयोग करके AI एजेंटों के लिए **टूल उपयोग** डिज़ाइन पैटर्न सीखेंगे। हम कवर करते हैं:

- `@tool` डेकोरेटर और टाइप किए गए पैरामीटर के साथ फ़ंक्शन टूल को परिभाषित करना
- यह बताने के लिए टूल स्कीमाएँ प्रदान करना कि प्रत्येक टूल क्या करता है
- `approval_mode` के साथ टूल निष्पादन को नियंत्रित करना
- Pydantic मॉडल और `response_format` के माध्यम से **संरचित आउटपुट** लौटाना

परिदृश्य एक **ट्रैवल बुकिंग एजेंट** है जो गंतव्यों को देख सकता है, उपलब्धता जांच सकता है, और फ्लाइट जानकारी प्राप्त कर सकता है।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool डेकोरेटर के साथ टूल्स को परिभाषित करना

`@tool` डेकोरेटर एक साधारण पायथन फ़ंक्शन को एक टूल में बदल देता है जिसे एक एजेंट कॉल कर सकता है।  
मुख्य बिंदु:

- **डॉकस्ट्रिंग** मॉडल द्वारा देखी जाने वाली टूल विवरण बन जाती है।  
- **टाइप एनोटेशन** (जिसमें विवरण के साथ `Annotated` शामिल हैं) टूल स्कीमा को परिभाषित करते हैं।  
- `approval_mode` यह नियंत्रित करता है कि क्या प्रत्येक कॉल को निष्पादित करने से पहले उपयोगकर्ता को अनुमोदन करना होगा।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## एक एजेंट कई उपकरणों के साथ बनाना

सभी तीन उपकरण क्लाइंट को पास करें ताकि मॉडल उपयोगकर्ता के प्रश्न का उत्तर देने के लिए जिन उपकरणों की आवश्यकता हो उन्हें कॉल कर सके।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## उपकरणों के साथ संरचित आउटपुट

`response_format` को Pydantic मॉडल पर सेट करने से, एजेंट को स्वतंत्र रूप से लिखे गए पाठ के बजाय एक अच्छी तरह प्रकारबद्ध JSON ऑब्जेक्ट लौटाने के लिए बाध्य किया जाता है। जब डाउनस्ट्रीम कोड को परिणाम को प्रोग्रामगत रूप से उपयोग करने की आवश्यकता होती है, तब यह उपयोगी होता है।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## टूल अनुमोदन पैटर्न

`@tool` पर `approval_mode` पैरामीटर यह नियंत्रित करता है कि टूल कॉल्स को निष्पादित करने से पहले मानव अनुमोदन की आवश्यकता है या नहीं:

| मोड | व्यवहार |
|---|---|
| `"never_require"` | टूल अपने आप चलता है — उपयोगकर्ता की पुष्टि की आवश्यकता नहीं है। |
| `"always_require"` | प्रत्येक कॉल को निष्पादित करने से पहले उपयोगकर्ता द्वारा अनुमोदित किया जाना चाहिए। |

ऐसे टूल्स के लिए `"always_require"` का उपयोग करें जिनके साइड-इफेक्ट्स होते हैं (जैसे फ्लाइट बुक करना, क्रेडिट कार्ड चार्ज करना) ताकि मानव प्रक्रिया में बना रहे।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## सारांश

इस पाठ में आपने सीखा कि कैसे:

1. टाइप किए गए पैरामीटर और डॉकस्ट्रिंग्स के साथ `@tool` डेकोरेटर का उपयोग करके **उपकरण परिभाषित करें** जो उपकरण स्कीमा के रूप में सेवा करते हैं।
2. **कई उपकरणों को संयोजित करें** ताकि एजेंट अनुप्रेषित प्रश्नों का उत्तर देने के लिए उन्हें अनुक्रम में कॉल कर सके।
3. **संरचित आउटपुट लौटाएं** Pydantic मॉडल को `response_format` के रूप में पास करके।
4. संवेदनशील ऑपरेशनों के लिए मानव शामिल रखने के लिए `approval_mode` के साथ **उपकरण अनुमोदन को नियंत्रित करें**।

ये पैटर्न बाहरी प्रणालियों के साथ सुरक्षित रूप से बातचीत करने वाले विश्वसनीय, उत्पादन-तैयार एजेंट बनाने की नींव बनाते हैं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
